# Semi-Supervised vs Supervised Comparison

**8 configurations**: RF · XGBoost · LSTM · Transformer  ×  Supervised · Semi-supervised

**Sections**
1. Full results table — F2, precision, recall, Brier, AUC for all 8 configs
2. Delta analysis — semi-supervised minus supervised F2, with CI overlap flag
3. Per-park breakdown — F2 heatmaps for all 8 configurations
4. Discussion — where semi-supervised helps and where it does not

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src" / "models"))

from evaluate import THREAT_LABELS

RESULTS = ROOT / "results"
(RESULTS / "plots").mkdir(parents=True, exist_ok=True)

THREAT_SHORT = {
    "fire_within_30d":       "Fire",
    "drought_within_30d":    "Drought",
    "vegetation_within_30d": "Vegetation",
}
PARK_LABELS = {
    "chad_basin":    "Chad Basin",
    "cross_river":   "Cross River",
    "gashaka_gumti": "Gashaka-Gumti",
    "kainji_lake":   "Kainji Lake",
    "old_oyo":       "Old Oyo",
    "yankari":       "Yankari",
}

ARCH_COLORS = {
    "RF":          "#2196F3",
    "XGBoost":     "#FF5722",
    "LSTM":        "#4CAF50",
    "Transformer": "#9C27B0",
}
REGIME_HATCH = {"Supervised": "", "Semi": "///"}

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("Setup complete.")

In [ ]:
result_files = {
    ("RF",          "Supervised"): "rf_supervised_results.json",
    ("RF",          "Semi"):       "rf_self_results.json",
    ("XGBoost",     "Supervised"): "xgb_supervised_results.json",
    ("XGBoost",     "Semi"):       "xgb_self_results.json",
    ("LSTM",        "Supervised"): "lstm_supervised_results.json",
    ("LSTM",        "Semi"):       "lstm_fixmatch_results.json",
    ("Transformer", "Supervised"): "transformer_supervised_results.json",
    ("Transformer", "Semi"):       "transformer_fixmatch_results.json",
}

res = {}
missing = []
for key, fname in result_files.items():
    path = RESULTS / fname
    if path.exists():
        with open(path) as f:
            res[key] = json.load(f)
        arch, regime = key
        print(f"{arch:12s} {regime:12s}  test mean-F2: {res[key]['test_mean_f2']:.4f}")
    else:
        missing.append(fname)

if missing:
    print(f"\nMissing results (run training scripts first):\n  " + "\n  ".join(missing))

## 1. Full Results Table

All 8 configurations on the held-out test set (Jan 2024 onward) at threshold 0.50.

In [ ]:
ARCHS   = ["RF", "XGBoost", "LSTM", "Transformer"]
REGIMES = ["Supervised", "Semi"]

rows = []
for arch in ARCHS:
    for regime in REGIMES:
        key = (arch, regime)
        if key not in res:
            continue
        for label in THREAT_LABELS:
            m  = res[key]["test_metrics"][label]
            ci = m["bootstrap_f2"]
            rows.append({
                "Architecture": arch,
                "Regime":       regime,
                "Threat":       THREAT_SHORT[label],
                "F2":           round(m["f2"], 4),
                "95% CI":       f"[{ci['ci_lower']:.4f}, {ci['ci_upper']:.4f}]",
                "Precision":    round(m["precision"], 4),
                "Recall":       round(m["recall"], 4),
                "Brier":        round(m["brier"], 4),
                "ROC-AUC":      round(m["roc_auc"], 4),
                "Persist F2":   round(m["persistence_f2"], 4),
                "Beats Base":   "Yes" if m["beats_baseline"] else "No",
            })

df_all = pd.DataFrame(rows).set_index(["Architecture", "Regime", "Threat"])
df_all

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

n_groups = len(ARCHS)
n_bars   = 2  # supervised + semi
group_w  = 0.7
bar_w    = group_w / n_bars

for ax, label in zip(axes, THREAT_LABELS):
    for gi, arch in enumerate(ARCHS):
        for ri, regime in enumerate(REGIMES):
            key = (arch, regime)
            if key not in res:
                continue
            m  = res[key]["test_metrics"][label]
            ci = m["bootstrap_f2"]
            f2 = m["f2"]
            x  = gi + (ri - 0.5) * bar_w
            lo = f2 - ci["ci_lower"]
            hi = ci["ci_upper"] - f2
            ax.bar(x, f2, bar_w * 0.9,
                   color=ARCH_COLORS[arch],
                   alpha=0.9 if regime == "Supervised" else 0.55,
                   hatch=REGIME_HATCH[regime],
                   edgecolor="black", linewidth=0.5)
            ax.errorbar(x, f2, yerr=[[lo], [hi]], fmt="none",
                        color="black", capsize=3, linewidth=1)

    persist = res.get(("RF", "Supervised"), {}).get("test_metrics", {}).get(label, {}).get("persistence_f2", None)
    if persist is not None:
        ax.axhline(persist, color="gray", linestyle="--", linewidth=1)
    ax.set_title(THREAT_SHORT[label], fontweight="bold")
    ax.set_xticks(range(n_groups))
    ax.set_xticklabels(ARCHS, rotation=15, ha="right")
    ax.set_ylim(0, 1.12)
    if ax is axes[0]:
        ax.set_ylabel("F2 Score")

handles = (
    [mpatches.Patch(color=ARCH_COLORS[a], label=a) for a in ARCHS] +
    [mpatches.Patch(facecolor="white", edgecolor="black", label="Supervised"),
     mpatches.Patch(facecolor="white", edgecolor="black", hatch="///", label="Semi-supervised"),
     plt.Line2D([0],[0], color="gray", linestyle="--", label="Persistence")]
)
fig.legend(handles=handles, loc="lower center", ncol=7, bbox_to_anchor=(0.5, -0.08))
fig.suptitle("F2 Score — Supervised vs Semi-supervised (test set, 95% bootstrap CI)",
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / "plots" / "f2_semi_vs_supervised.png", bbox_inches="tight")
plt.show()

## 2. Delta Analysis

Delta = semi-supervised F2 minus supervised F2.  
**Distinguishable** = the two bootstrap 95% CIs do not overlap.

In [ ]:
def cis_overlap(ci_a, ci_b) -> bool:
    """True if two [lower, upper] CI intervals overlap."""
    return ci_a["ci_lower"] <= ci_b["ci_upper"] and ci_b["ci_lower"] <= ci_a["ci_upper"]

delta_rows = []
for arch in ARCHS:
    sup_key  = (arch, "Supervised")
    semi_key = (arch, "Semi")
    if sup_key not in res or semi_key not in res:
        continue
    for label in THREAT_LABELS:
        m_sup  = res[sup_key]["test_metrics"][label]
        m_semi = res[semi_key]["test_metrics"][label]
        delta  = m_semi["f2"] - m_sup["f2"]
        overlap = cis_overlap(m_sup["bootstrap_f2"], m_semi["bootstrap_f2"])
        delta_rows.append({
            "Architecture": arch,
            "Threat":       THREAT_SHORT[label],
            "Supervised F2": round(m_sup["f2"], 4),
            "Semi F2":       round(m_semi["f2"], 4),
            "Delta F2":      round(delta, 4),
            "Distinguishable": "No" if overlap else "Yes",
        })

df_delta = pd.DataFrame(delta_rows).set_index(["Architecture", "Threat"])
df_delta.style.background_gradient(
    subset=["Delta F2"], cmap="RdYlGn", vmin=-0.05, vmax=0.05
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

n_threats = len(THREAT_LABELS)
x         = np.arange(n_threats)
w         = 0.18
offsets   = [-1.5, -0.5, 0.5, 1.5]

for j, arch in enumerate(ARCHS):
    sup_key  = (arch, "Supervised")
    semi_key = (arch, "Semi")
    if sup_key not in res or semi_key not in res:
        continue
    deltas = [
        res[semi_key]["test_metrics"][l]["f2"] - res[sup_key]["test_metrics"][l]["f2"]
        for l in THREAT_LABELS
    ]
    bars = ax.bar(x + offsets[j] * w, deltas, w,
                  color=ARCH_COLORS[arch], alpha=0.85, label=arch)
    for bar, d in zip(bars, deltas):
        if abs(d) > 0.002:
            va = "bottom" if d >= 0 else "top"
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + (0.001 if d >= 0 else -0.001),
                    f"{d:+.3f}", ha="center", va=va, fontsize=7)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([THREAT_SHORT[l] for l in THREAT_LABELS])
ax.set_ylabel("F2 delta (semi − supervised)")
ax.set_title("Semi-supervised Gain / Loss per Architecture and Threat", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / "plots" / "semi_vs_supervised_delta.png", bbox_inches="tight")
plt.show()

## 3. Per-Park Breakdown

Supervised F2 (top row) vs semi-supervised F2 (bottom row) for each architecture.

In [ ]:
park_rows = []
for arch in ARCHS:
    for regime in REGIMES:
        key = (arch, regime)
        if key not in res:
            continue
        for park, pm in res[key]["per_park_test"].items():
            row = {"Architecture": arch, "Regime": regime, "Park": PARK_LABELS.get(park, park)}
            for label in THREAT_LABELS:
                row[THREAT_SHORT[label]] = round(pm[label]["f2"], 3)
            park_rows.append(row)

df_park = pd.DataFrame(park_rows).set_index(["Architecture", "Regime", "Park"])
df_park

In [ ]:
# Delta per-park heatmap: semi - supervised
available = [(a, "Supervised") in res and (a, "Semi") in res for a in ARCHS]
archs_avail = [a for a, ok in zip(ARCHS, available) if ok]

if archs_avail:
    fig, axes = plt.subplots(1, len(archs_avail), figsize=(4 * len(archs_avail), 5))
    if len(archs_avail) == 1:
        axes = [axes]

    for ax, arch in zip(axes, archs_avail):
        sup_pp  = res[(arch, "Supervised")]["per_park_test"]
        semi_pp = res[(arch, "Semi")]["per_park_test"]
        parks   = list(sup_pp.keys())
        data = np.array([
            [semi_pp[p][l]["f2"] - sup_pp[p][l]["f2"] for l in THREAT_LABELS]
            for p in parks
        ])
        park_disp  = [PARK_LABELS.get(p, p) for p in parks]
        threat_disp = [THREAT_SHORT[l] for l in THREAT_LABELS]

        lim = max(0.01, np.abs(data).max())
        im  = ax.imshow(data, vmin=-lim, vmax=lim, cmap="RdYlGn", aspect="auto")
        ax.set_xticks(range(3))
        ax.set_xticklabels(threat_disp)
        ax.set_yticks(range(len(parks)))
        ax.set_yticklabels(park_disp)
        ax.set_title(f"{arch}  Δ F2 (semi − sup)", fontweight="bold",
                     color=ARCH_COLORS[arch])
        for i in range(len(parks)):
            for j in range(3):
                v  = data[i, j]
                tc = "black"
                ax.text(j, i, f"{v:+.2f}", ha="center", va="center",
                        color=tc, fontsize=8)
        plt.colorbar(im, ax=ax, label="Δ F2")

    plt.tight_layout()
    plt.savefig(RESULTS / "plots" / "per_park_delta_heatmap.png", bbox_inches="tight")
    plt.show()

## 4. Discussion

### Where semi-supervised helps
Self-training for tree models (RF, XGBoost) can improve performance when the unlabeled
pool is large and the initial supervised model is already well-calibrated — high-confidence
pseudo-labels effectively expand the training set without introducing noise.

FixMatch for neural models adds a consistency regulariser that can reduce overfitting on
small labeled datasets; the augmentation noise (5% weak / 15% strong) is small enough
that pseudo-labels from weak augmentations are reliable starting points.

### Where semi-supervised does not help
If the unlabeled pool (ndvi_missing==1 rows) is small relative to the labeled set, the
effect of self-training is negligible — few rows meet the all-threat confidence gate
(0.90 / 0.95), and the additional data does not shift the decision boundary.

For the Transformer, which already showed early overfitting under supervised training
(epoch 18), FixMatch adds an unsupervised loss on top of an already-noisy gradient
signal; this can hurt if pseudo-label confidence is low in early epochs.

### Interpretation
The delta column marks results as **distinguishable** only when the bootstrap 95% CIs of
supervised and semi-supervised F2 do not overlap.  Non-overlapping CIs are a conservative
criterion; a small positive delta that falls within the CI width should not be interpreted
as a reliable improvement.

In [ ]:
# Summary table: mean-F2 for all 8 configurations
summary_rows = []
for arch in ARCHS:
    for regime in REGIMES:
        key = (arch, regime)
        if key not in res:
            continue
        tm  = res[key]["test_metrics"]
        row = {
            "Architecture": arch,
            "Regime":       regime,
            "Mean F2":      round(res[key]["test_mean_f2"], 4),
        }
        for label in THREAT_LABELS:
            row[THREAT_SHORT[label] + " F2"] = round(tm[label]["f2"], 4)
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index(["Architecture", "Regime"])
print(df_summary.to_string())

## 5. Accuracy, Confusion Matrices & Classification Reports

Accuracy per threat for all 8 configurations.
Confusion matrices: one figure per threat, top row = supervised, bottom row = semi-supervised.
Class 1 = event present within 30 days; Class 0 = no event.

In [ ]:
acc_rows = []
for arch in ARCHS:
    for regime in REGIMES:
        key = (arch, regime)
        if key not in res:
            continue
        row = {"Architecture": arch, "Regime": regime}
        for label in THREAT_LABELS:
            row[THREAT_SHORT[label]] = round(
                res[key]["test_metrics"][label]["accuracy"], 4
            )
        acc_rows.append(row)

df_acc = pd.DataFrame(acc_rows).set_index(["Architecture", "Regime"])
df_acc

In [ ]:
# One figure per threat: 2 rows (Supervised / Semi) × 4 cols (architectures)
for label in THREAT_LABELS:
    fig, axes = plt.subplots(2, 4, figsize=(14, 5))
    fig.suptitle(f"Confusion Matrices — {THREAT_SHORT[label]} (threshold 0.50)",
                 fontweight="bold", y=1.02)

    for ri, regime in enumerate(REGIMES):
        for ci, arch in enumerate(ARCHS):
            ax  = axes[ri, ci]
            key = (arch, regime)
            if key not in res:
                ax.axis("off")
                continue
            cm_d = res[key]["test_metrics"][label]["confusion_matrix"]
            cm   = np.array([[cm_d["tn"], cm_d["fp"]], [cm_d["fn"], cm_d["tp"]]])

            ax.imshow(cm, cmap="Blues", interpolation="nearest")
            ax.set_xticks([0, 1])
            ax.set_yticks([0, 1])
            ax.set_xticklabels(["Pred 0", "Pred 1"], fontsize=7)
            ax.set_yticklabels(["True 0", "True 1"], fontsize=7)

            thresh = cm.max() / 2
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                            fontsize=9, fontweight="bold",
                            color="white" if cm[i, j] > thresh else "black")

            title = arch if ri == 0 else ""
            if title:
                ax.set_title(title, fontweight="bold", color=ARCH_COLORS[arch])
            if ci == 0:
                ax.set_ylabel(regime, fontweight="bold")

    plt.tight_layout()
    fname = f"confusion_matrices_{THREAT_SHORT[label].lower()}_semi_vs_sup.png"
    plt.savefig(RESULTS / "plots" / fname, bbox_inches="tight")
    plt.show()

In [ ]:
cr_rows = []
for arch in ARCHS:
    for regime in REGIMES:
        key = (arch, regime)
        if key not in res:
            continue
        for label in THREAT_LABELS:
            m  = res[key]["test_metrics"][label]
            cr = m["classification_report"]
            cr_rows.append({
                "Architecture": arch,
                "Regime":       regime,
                "Threat":       THREAT_SHORT[label],
                "Accuracy":     round(m["accuracy"], 4),
                "Prec (0)":     round(cr["0"]["precision"], 4),
                "Recall (0)":   round(cr["0"]["recall"], 4),
                "F1 (0)":       round(cr["0"]["f1-score"], 4),
                "Prec (1)":     round(cr["1"]["precision"], 4),
                "Recall (1)":   round(cr["1"]["recall"], 4),
                "F1 (1)":       round(cr["1"]["f1-score"], 4),
                "Macro F1":     round(cr["macro avg"]["f1-score"], 4),
            })

df_cr05 = pd.DataFrame(cr_rows).set_index(["Architecture", "Regime", "Threat"])
df_cr05